# Chapter 5: The JSON Config Approach

The central idea of this book: **scraper configurations expressed as JSON are a form of structured language that both humans and AI agents can read, write, and reason about.**

This notebook shows:
1. How to read and use the config schema
2. How to build a config step by step for our demo sites
3. How the config drives the scraper engine
4. Why this matters for autonomous AI scraping

In [8]:
import json
import re
import httpx
from bs4 import BeautifulSoup
from pathlib import Path

## 5.1 The Config Schema

A scraper config is a JSON object with four top-level sections:

```
{
  "sources"  : WHERE to fetch (URLs + pagination)
  "listing"  : HOW to find detail page links on the listing page
  "fields"   : WHAT to extract from each detail page
  "render_mode": static | playwright | auto
}
```

In [9]:
# Load the pre-built ShopSphere SSR config
config_path = Path("../configs/shopsphere-ssr.json")
with open(config_path) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "name": "ShopSphere SSR",
  "description": "Server-side rendered marketplace. Static fetch works directly.",
  "render_mode": "static",
  "request": {
    "headers": {
      "User-Agent": "Mozilla/5.0 (compatible; AutonomousScraper/1.0)"
    }
  },
  "sources": [
    {
      "url_template": "http://localhost:8001/products?page={n}",
      "index_type": "page",
      "pagination": {
        "start": 1,
        "step": 1,
        "max_pages": 10,
        "stop_condition": "no_results"
      }
    }
  ],
  "listing": {
    "link_selector": "a.product-link",
    "link_prefix": "http://localhost:8001"
  },
  "fields": {
    "title": {
      "selector": "h1.product-title",
      "retrieve": "plaintext"
    },
    "price": {
      "selector": "span.price-amount",
      "retrieve": "regexp",
      "pattern": "[\\d,.]+"
    },
    "rating": {
      "selector": "span.rating-value",
      "retrieve": "attr",
      "attr": "data-rating"
    },
    "review_count": {
      "selector": "span.revi

## 5.2 Building a Config from Scratch

In [10]:
# Step 1: Examine the listing page
listing_html = httpx.get("http://localhost:8003/jobs?page=1").text
listing_soup = BeautifulSoup(listing_html, "lxml")

job_links = listing_soup.select("a.job-link")
print(f"Link selector 'a.job-link' matches: {len(job_links)} elements")
for link in job_links[:3]:
    print(f"  {link.get_text(strip=True)[:40]} -> {link['href']}")

Link selector 'a.job-link' matches: 10 elements
  Senior Backend Engineer -> /jobs/senior-backend-engineer-stripe
  Staff Software Engineer -> /jobs/staff-software-engineer-airbnb
  Senior Frontend Engineer -> /jobs/senior-frontend-engineer-figma


In [11]:
# Step 2: Test selectors on a detail page
first_job_url = "http://localhost:8003" + job_links[0]["href"]
detail_html = httpx.get(first_job_url).text
detail_soup = BeautifulSoup(detail_html, "lxml")

selectors_to_test = {
    "title": "h1.job-detail-title",
    "company": "div.job-detail-company",
    "location": "span.job-location",
    "salary": "span.salary-range",
    "remote_type": "span.badge-remote",
    "description": "p.job-description-full",
    "requirements": "li.requirement-item",
}

print(f"{'Field':15} {'Matches':>8} {'Value preview'}")
print("-" * 70)
for field, selector in selectors_to_test.items():
    els = detail_soup.select(selector)
    val = els[0].get_text(strip=True)[:50] if els else "[NO MATCH]"
    print(f"{field:15} {len(els):>8} {val!r}")

Field            Matches Value preview
----------------------------------------------------------------------
title                  1 'Senior Backend Engineer'
company                1 'Stripe'
location               1 'San Francisco, CA'
salary                 1 '$180,000 - $240,000'
remote_type            1 'Hybrid'
description            1 'Build the financial infrastructure of the internet'
requirements           5 'Python or Go'


In [12]:
# Step 3: Build the complete config
jobhive_config = {
    "name": "JobHive SSR (built in notebook)",
    "render_mode": "static",
    "sources": [{
        "url_template": "http://localhost:8003/jobs?page={n}",
        "pagination": {"start": 1, "step": 1, "max_pages": 10, "stop_condition": "no_results"}
    }],
    "listing": {"link_selector": "a.job-link", "link_prefix": "http://localhost:8003"},
    "fields": {
        "title": {"selector": "h1.job-detail-title", "retrieve": "plaintext"},
        "company": {"selector": "div.job-detail-company", "retrieve": "plaintext"},
        "location": {"selector": "span.job-location", "retrieve": "plaintext"},
        "salary_min": {"selector": "span.salary-range", "retrieve": "regexp", "pattern": "\\$(\\d[\\d,]+)"},
        "salary_max": {"selector": "span.salary-range", "retrieve": "regexp", "pattern": "\\$[\\d,]+ - \\$(\\d[\\d,]+)"},
        "remote_type": {"selector": "span.badge-remote", "retrieve": "plaintext"},
        "job_type": {"selector": "span.badge-type", "retrieve": "plaintext"},
        "description": {"selector": "p.job-description-full", "retrieve": "plaintext"},
        "job_id": {"selector": "div.job-detail", "retrieve": "attr", "attr": "data-job-id"},
        "requirements": {"selector": "li.requirement-item", "retrieve": "plaintext", "multiple": True},
        "benefits": {"selector": "li.benefit-item", "retrieve": "plaintext", "multiple": True},
    }
}

print("Config built. Fields:", list(jobhive_config["fields"].keys()))

Config built. Fields: ['title', 'company', 'location', 'salary_min', 'salary_max', 'remote_type', 'job_type', 'description', 'job_id', 'requirements', 'benefits']


## 5.3 Executing the Config

In [13]:
def extract_field(soup, field_cfg):
    selector = field_cfg.get("selector", "")
    retrieve = field_cfg.get("retrieve", "plaintext")
    multiple = field_cfg.get("multiple", False)
    attr = field_cfg.get("attr", "")
    pattern = field_cfg.get("pattern", "")
    
    elements = soup.select(selector) if selector else []
    if not elements:
        return None
    
    def get_val(el):
        if retrieve == "attr":
            return el.get(attr)
        text = el.get_text(separator=" ", strip=True)
        if retrieve == "regexp" and pattern:
            m = re.search(pattern, text)
            return m.group(1) if m and m.lastindex else (m.group(0) if m else None)
        return text
    
    return [get_val(el) for el in elements] if multiple else get_val(elements[0])


def run_config(config, max_items=3):
    results = []
    listing = config["listing"]
    link_sel = listing["link_selector"]
    prefix = listing.get("link_prefix", "")
    
    for source in config["sources"]:
        tmpl = source["url_template"]
        p = source["pagination"]
        for i in range(p["max_pages"]):
            page_url = tmpl.replace("{n}", str(p["start"] + i * p["step"]))
            soup = BeautifulSoup(httpx.get(page_url, headers={"User-Agent": "Mozilla/5.0"}).text, "lxml")
            links = soup.select(link_sel)
            if not links:
                break
            for link_el in links:
                if len(results) >= max_items:
                    return results
                href = link_el.get("href", "")
                url = prefix.rstrip("/") + "/" + href.lstrip("/") if href.startswith("/") else href
                ds = BeautifulSoup(httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}).text, "lxml")
                record = {"source_url": url}
                for fname, fcfg in config["fields"].items():
                    record[fname] = extract_field(ds, fcfg)
                results.append(record)
    return results


results = run_config(jobhive_config, max_items=3)
for r in results:
    print(f"\n{r.get('title')} at {r.get('company')}")
    print(f"  Location: {r.get('location')} | Remote: {r.get('remote_type')}")
    print(f"  Salary: ${r.get('salary_min')} - ${r.get('salary_max')}")
    print(f"  Requirements: {r.get('requirements', [])[:3]}")


Senior Backend Engineer at Stripe
  Location: San Francisco, CA | Remote: Hybrid
  Salary: $180,000 - $240,000
  Requirements: ['Python or Go', 'Distributed systems', 'PostgreSQL']

Staff Software Engineer at Airbnb
  Location: Remote | Remote: Remote
  Salary: $200,000 - $280,000
  Requirements: ['Java or Kotlin', 'Large-scale distributed systems', 'Technical leadership']

Senior Frontend Engineer at Figma
  Location: San Francisco, CA | Remote: Hybrid
  Salary: $160,000 - $220,000
  Requirements: ['TypeScript', 'React', 'WebGL or Canvas']


## 5.4 Config as LLM-Readable Language

In [14]:
import sys
sys.path.insert(0, "../agents")
from llm import chat

# Ask the LLM to explain the config
messages = [
    {"role": "system", "content": "You are a scraping expert. Explain scraper configs in plain English, concisely (under 100 words)."},
    {"role": "user", "content": f"Explain what this scraper config does:\n\n{json.dumps(jobhive_config, indent=2)}"}
]
print(chat(messages, max_tokens=200))

This config scrapes job listings from a local JobHive site (running on `localhost:8003`).

- **Sources**: Scrapes pages 1–10 of `/jobs?page={n}` (stops if no results).
- **Listing**: Extracts job links from `<a class="job-link">`.
- **Fields**: Pulls job details like:
  - Basic info (title, company, location) from HTML elements.
  - Salary (min/max) using regex from text.
  - Remote type, job type, and description from plain text.
  - Job ID from a `data-job-id` attribute.
  - Lists requirements/benefits (multiple items) from `<li>` tags.

*Static* mode means it doesn’t render JavaScript.


## 5.5 Key Takeaways

1. **The JSON config is a complete scraping specification**: sources, pagination, selectors, render mode
2. **Building a config** follows a systematic process: inspect listing, find link selector, inspect detail, map each field
3. **The config engine separates intent from execution** - you declare WHAT to extract, the engine handles HOW
4. **LLMs can work with configs directly** - read, generate, modify, validate
5. **This creates a new paradigm**: instead of writing scraping code, you write configurations, and AI agents write those for you